pt 2 of exploration_analysis.

wil try to read df with 4 pps, split per groups and do cross val. 

if this works i can then look into some tuning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pykalman import KalmanFilter

In [4]:
#should read in some sort of processed df, exported from exploration analysis notebook. since im not doing it the next cells are just filling in NAs
df = pd.read_csv('all_data_4pps.csv')

In [6]:
   # Very simple Kalman filter: fill missing values and remove outliers for single attribute.
    # We assume a very simple transition matrix, namely simply a [[1]]. It
    # is however still useful as it is able to dampen outliers and impute missing values. The new
    # values are appended in a new column when replaceCol=False
# RETURNS THE SAME DF THAT IT RECEIVED AS INPUT-to make it work pp-wise
def apply_kalman_filter(df, col, replaceCol = False):

    # df = df2.copy()
    # df = df2#if u pass in sub-df, due to looping over pps dont copy (so that changes are also on the big main df)
    

    # Initialize the Kalman filter with the trivial transition and observation matrices.
    kf = KalmanFilter(transition_matrices=[[1]], observation_matrices=[[1]])

    numpy_array_state = df[col].values
    numpy_array_state = numpy_array_state.astype(np.float32)
    numpy_matrix_state_with_mask = np.ma.masked_invalid(numpy_array_state)

    # Find the best other parameters based on the data (e.g. Q)
    kf = kf.em(numpy_matrix_state_with_mask, n_iter=5)

    # And apply the filter.
    (new_data, filtered_state_covariances) = kf.filter(numpy_matrix_state_with_mask)

    if replaceCol:
        df[col] = new_data
    else:
        df[col + '_kalman'] = new_data
    return df


# #applying it to all num cols -NOT PP WISE SO WRONG
for col in df.select_dtypes(include='number').columns:
    df = apply_kalman_filter(df, col, replaceCol=True)
    

In [10]:
num_cols = ['AU1_InnerBrowRaiser', 'AU2_OuterBrowRaiser', 'AU4_BrowLowerer', 'AU5_UpperLipRaiser', 'AU6_CheekRaiser', 'AU7_LidTightener', 'AU9_NoseWringler', 'AU10_UpperLipRaiser', 'AU12_LipCornerPuller',
 'AU14_Dimpler','AU15_LipCornerDepressonr', 'AU17_ChinRaiser', 'AU20_LipStretcher', 'AU23_LipTightener', 'AU25_LipsPart', 'AU26_JawDrop', 'AU45_BlinkInt', 'AUc45_BlinkRate', 'AUc28_LipSuck',
 'headOrient_x', 'headOrient_y', 'headOrient_z', 'gazeCenter', 'gazeUp', 'gazeDown', 'gazeRight', 'gazeLeft', 'duration', 'Rclick', 'Lclick', 'move_dist', 'move_duration', 'move_speed',
 'scroll_dur', 'keyPress', 'press_dur', 'backsp', 'backsp_dur', 'pause_dur', 'pause_rate']

feature_cols = num_cols

## modeling

In [13]:
from sklearn.model_selection import train_test_split ,cross_val_score,RandomizedSearchCV, cross_validate, StratifiedKFold
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn import svm
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,f1_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### splitting per group, & CV (no tuning)

In [46]:
x = df[feature_cols] 
y = df['condition']

models = [svm.SVC( random_state=42), KNeighborsClassifier(), DecisionTreeClassifier(random_state=41), RandomForestClassifier(random_state=43)]

model = models[0]

PPgroups = df['pp_id'].copy().tolist()

#initial splitting, this test will be used for final testing. the train will be split further in CV
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=43)
for train_index, test_index in gss.split(x, y, groups=PPgroups):
    print('a test-train split')
    x_train = x.iloc[train_index, :]
    x_test = x.iloc[test_index, :]
    y_train = y[train_index]
    y_test = y[test_index]
    pp_groups_trainOnly = df.iloc[train_index]['pp_id'] #will have one pp less than PPgroups!
    print('done splitting')

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

scores = cross_val_score(model, x_train_scaled, y_train, groups = pp_groups_trainOnly, scoring='accuracy', cv=GroupKFold(n_splits=3))
print(model)
print('cross val scores: {}'.format(scores))

#then can train on full data



a test-train split
done splitting
SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='rbf',
    max_iter=-1, probability=False, random_state=42, shrinking=True, tol=0.001,
    verbose=False)
cross val scores: [0.65 0.7  0.5 ]


In [44]:
# checking pp distribution in train and test
for train_index, test_index in gss.split(x, y, groups = df['pp_id']): 
    print('a split:')
    print('train')
    print(set(df.iloc[train_index]['pp_id']))
    # print((df.iloc[train_index]['pp_id']))
    print('test')
    print(set(df.iloc[test_index]['pp_id'] ))
    # print((df.iloc[test_index]['pp_id'] ))

a split:
train
{'pp23', 'pp02', 'pp05'}
test
{'pp18'}


### SVM: splitting per group + CV tuning 

In [61]:
x = df[feature_cols] 
y = df['condition']

# models = [svm.SVC( random_state=42), KNeighborsClassifier(), DecisionTreeClassifier(random_state=41), RandomForestClassifier(random_state=43)]

# model = models[0]

PPgroups = df['pp_id'].copy().tolist()

#initial splitting, this test will be used for final testing. the train will be split further in CV
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=43)
for train_index, test_index in gss.split(x, y, groups=PPgroups):
    print('a test-train split')
    x_train = x.iloc[train_index, :]
    x_test = x.iloc[test_index, :]
    y_train = y[train_index]
    y_test = y[test_index]
    pp_groups_trainOnly = df.iloc[train_index]['pp_id'] #will have one(or some) pp less than PPgroups!
    print('done splitting')

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

######################################################### maybe make the above into function
# List of C values
C_range = np.logspace(-10, 10, 21)
# List of gamma values
gamma_range = np.logspace(-10, 10, 21)

# Define the search space
param_grid = { 
    # Regularization parameter.
    "C": C_range,
    # Kernel type
    "kernel": ['rbf', 'poly'],
    # Gamma is the Kernel coefficient for ‘rbf’, ‘poly’ and ‘sigmoid’.
    "gamma": gamma_range.tolist()+['scale', 'auto']
}
# Set up score
scoring = ['accuracy']

# Set up the k-fold cross-validation
gkfold = GroupKFold(n_splits = 2)

# Define random search
random_search_svm = RandomizedSearchCV(estimator= svm.SVC( random_state=42), 
                           param_distributions= param_grid, 
                           n_iter= 100,
                           scoring= ['accuracy'], 
                           refit= 'accuracy', 
                           n_jobs= -1, 
                           cv= GroupKFold(n_splits = 3), 
                           verbose=0,
                           random_state=42)

# Fit grid search
random_result_svm = random_search_svm.fit(x_train_scaled, y_train, groups= pp_groups_trainOnly) 

# print('printing_result of random search cv')
# random_result

a test-train split
done splitting
printing_result of random search cv


RandomizedSearchCV(cv=GroupKFold(n_splits=3), error_score=nan,
                   estimator=SVC(C=1.0, break_ties=False, cache_size=200,
                                 class_weight=None, coef0=0.0,
                                 decision_function_shape='ovr', degree=3,
                                 gamma='scale', kernel='rbf', max_iter=-1,
                                 probability=False, random_state=42,
                                 shrinking=True, tol=0.001, verbose=False),
                   iid='deprecated', n_iter=10, n_jobs=-1,
                   param_distributions=...
       1.e+06, 1.e+07, 1.e+08, 1.e+09, 1.e+10]),
                                        'gamma': [1e-10, 1e-09, 1e-08, 1e-07,
                                                  1e-06, 1e-05, 0.0001, 0.001,
                                                  0.01, 0.1, 1.0, 10.0, 100.0,
                                                  1000.0, 10000.0, 100000.0,
                                          

In [62]:
print(f'The best accuracy score for the training dataset is {random_result.best_score_:.4f}')

print(f'The best hyperparameters are {random_result.best_params_}')
# Print the best accuracy score for the testing dataset
print(f'The accuracy score for the testing dataset is {random_search.score(x_test_scaled, y_test):.4f}') #!!!!!!!!!!!!!!!!!!!!!
print('printing results')
# random_result.cv_results_
random_result.cv_results_['params'] #the params used

The best accuracy score for the training dataset is 0.6222
The best hyperparameters are {'kernel': 'rbf', 'gamma': 1e-06, 'C': 100000000.0}
The accuracy score for the testing dataset is 0.4167
printing results


[{'kernel': 'rbf', 'gamma': 0.001, 'C': 1e-05},
 {'kernel': 'poly', 'gamma': 1e-07, 'C': 1.0},
 {'kernel': 'rbf', 'gamma': 1e-06, 'C': 100000000.0},
 {'kernel': 'poly', 'gamma': 1e-08, 'C': 100.0},
 {'kernel': 'rbf', 'gamma': 100.0, 'C': 1e-09},
 {'kernel': 'rbf', 'gamma': 1000000000.0, 'C': 0.0001},
 {'kernel': 'poly', 'gamma': 1e-10, 'C': 10000000000.0},
 {'kernel': 'poly', 'gamma': 1e-08, 'C': 10000000.0},
 {'kernel': 'rbf', 'gamma': 'scale', 'C': 1e-09},
 {'kernel': 'poly', 'gamma': 1.0, 'C': 10000.0}]

### RF: splitting per group + CV tuning (on data already split+scaled above)

In [65]:
# i already have x_train, y train (2b used for random search cv) and x test, y test

params_rf={'n_estimators': [100, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000],
                  'criterion':['gini','entropy'],
                  'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, None]}

random_search_rf = RandomizedSearchCV(estimator= RandomForestClassifier(random_state=43), 
                           param_distributions= params_rf, 
                           n_iter= 100,
                           scoring= ['accuracy'], 
                           refit= 'accuracy', 
                           n_jobs= -1, 
                           cv= GroupKFold(n_splits = 3), 
                           verbose=0,
                           random_state=42)

# Fit grid search
random_result_rf = random_search_rf.fit(x_train_scaled, y_train, groups= pp_groups_trainOnly) 

print(f'The best accuracy score for the training dataset is {random_search_rf.best_score_:.4f}')

print(f'The best hyperparameters are {random_result_rf.best_params_}')
# Print the best accuracy score for the testing dataset
print(f'The accuracy score for the testing dataset is {random_search_rf.score(x_test_scaled, y_test):.4f}') #!!!!!!!!!!!!!!!!!!!!!
# print('printing results')
# random_result.cv_results_
# random_result_rf.cv_results_['params'] #the params used
 

The best accuracy score for the training dataset is 0.6722
The best hyperparameters are {'n_estimators': 400, 'max_depth': 30, 'criterion': 'gini'}
The accuracy score for the testing dataset is 0.5000
printing results


### DT: splitting per group + CV tuning (on data already split+scaled above)

4 tree graph: https://www.kaggle.com/code/gauravduttakiit/hyperparameter-tuning-in-decision-trees

In [67]:


# i already have x_train, y train (2b used for random search cv) and x test, y test

params_dt={
    'max_depth': [2, 3, 5, 10, 20, 50, 100, 150, 200, 300, 500],
    'min_samples_leaf': [5, 10, 20, 50, 100, 200, 500],
    'criterion': ["gini", "entropy"]}


random_search_dt = RandomizedSearchCV(estimator= DecisionTreeClassifier(random_state=41), 
                           param_distributions= params_dt, 
                           n_iter= 100,
                           scoring= ['accuracy'], 
                           refit= 'accuracy', 
                           n_jobs= -1, 
                           cv= GroupKFold(n_splits = 3), 
                           verbose=0,
                           random_state=42)

# Fit grid search
random_result_dt = random_search_dt.fit(x_train_scaled, y_train, groups= pp_groups_trainOnly) 

print(f'The best accuracy score for the training dataset is {random_search_dt.best_score_:.4f}')

print(f'The best hyperparameters are {random_result_dt.best_params_}')
# Print the best accuracy score for the testing dataset
print(f'The accuracy score for the testing dataset is {random_search_dt.score(x_test_scaled, y_test):.4f}') #!!!!!!!!!!!!!!!!!!!!!
# print('printing results')
# random_result.cv_results_
# random_result_rf.cv_results_['params'] #the params used
 

The best accuracy score for the training dataset is 0.7500
The best hyperparameters are {'min_samples_leaf': 5, 'max_depth': 2, 'criterion': 'gini'}
The accuracy score for the testing dataset is 0.5000


### K Nearest Neighbors: splitting per group + CV tuning (on data already split+scaled above)

https://www.kaggle.com/code/arunimsamudra/k-nn-with-hyperparameter-tuning

In [74]:

# i already have x_train, y train (2b used for random search cv) and x test, y test

params_knn={ 'n_neighbors' : list(range(1, 31)),
               'weights' : ['uniform','distance'],
               'metric' : ['minkowski','euclidean','manhattan']}


random_search_knn = RandomizedSearchCV(estimator= KNeighborsClassifier(), 
                           param_distributions= params_knn, 
                           n_iter= 100,
                           scoring= ['accuracy'], 
                           refit= 'accuracy', 
                           n_jobs= -1, 
                           cv= GroupKFold(n_splits = 3), 
                           verbose=0,
                           random_state=42)

# Fit grid search
random_result_knn = random_search_knn.fit(x_train_scaled, y_train, groups= pp_groups_trainOnly) 

print(f'The best accuracy score for the training dataset is {random_search_knn.best_score_:.4f}')

print(f'The best hyperparameters are {random_result_knn.best_params_}')
# Print the best accuracy score for the testing dataset
print(f'The accuracy score for the testing dataset is {random_search_knn.score(x_test_scaled, y_test):.4f}') #!!!!!!!!!!!!!!!!!!!!!
# print('printing results')
# random_result_knn.cv_results_
# random_result_knn.cv_results_['params'] #the params used
 

The best accuracy score for the training dataset is 0.5833
The best hyperparameters are {'weights': 'distance', 'n_neighbors': 3, 'metric': 'euclidean'}
The accuracy score for the testing dataset is 0.5000
